# Geometric Compactness Optimization — Greedy Assembly of Polyominoes

Python implementation of the article:
  "Discrete Optimization of Geometric Compactness:
   A Dual Heuristic Approach to 2D Tiling"

Three heuristics are implemented and compared:
  H1 — Maximum order (fast greedy, O(V))
  H2 — Order-product distance (adaptive, O(V²))
  H3 — Greedy brute force (absolute baseline)

Each heuristic is run through a multi-branch beam search
in order to avoid local optima.

Two resemblance metrics are used to compare the solutions obtained by H1 and H2
morphologically against H3:
  R_simple   = N_diff / A_tot
  R_weighted = Σ w_i / A_tot   (with w_i = eccentricity degree)

## Usage

  ### Run with default parameters
  python compacite_gloutonne.py

  ### From a notebook (Kaggle / Colab / Jupyter)
  main(n_shapes=50, seed=7)

Author  : [Garrido Andersson Karl]
Date    : 04/2026

# ALGORITHM

## LIBRARIES

In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass
from typing import Optional

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba

## TYPE ALIAS

In [ ]:
Cell = tuple[int, int]

## BASE STRUCTURE

In [ ]:
@dataclass
class Shape:
    """
    Represents a block (polyomino): a connected set of primary units.

    Attributes
    ----------
    id    : unique block identifier
    cells : set (frozenset) of the cells (x, y) making up the block
    color : visualization color (hex string)

    Computed properties
    -------------------
    area        : number of cells (area)
    perimeter   : number of exposed edges (discrete perimeter)
    compactness : c(f) = area / perimeter  (Definition 3 of the article)
    """

    id: int
    cells: frozenset[Cell]
    color: str = "#4ECDC4"

    def __post_init__(self) -> None:
        self.cells = frozenset(self.cells)

    # ── Metrics ───────────────────────────────────────────────────────────

    @property
    def area(self) -> int:
        return len(self.cells)

    @property
    def perimeter(self) -> int:
        p = 0
        for x, y in self.cells:
            for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                if (x + dx, y + dy) not in self.cells:
                    p += 1
        return p

    @property
    def compactness(self) -> float:
        """c(f) = A / P  (Definition 3). Returns 0 if the perimeter is zero."""
        p = self.perimeter
        return self.area / p if p > 0 else 0.0

    # ── Geometric transformations ─────────────────────────────────────────

    def normalized(self) -> Shape:
        """Translates so that min(x) = 0 and min(y) = 0."""
        if not self.cells:
            return self
        min_x = min(x for x, _ in self.cells)
        min_y = min(y for _, y in self.cells)
        return Shape(
            self.id,
            frozenset((x - min_x, y - min_y) for x, y in self.cells),
            self.color,
        )

    def translated(self, dx: int, dy: int) -> Shape:
        """Returns a copy translated by (dx, dy)."""
        return Shape(
            self.id,
            frozenset((x + dx, y + dy) for x, y in self.cells),
            self.color,
        )

    def rotated(self, n: int) -> Shape:
        """Returns a copy after n clockwise 90° rotations, normalized."""
        cells = list(self.cells)
        for _ in range(n % 4):
            cells = [(y, -x) for x, y in cells]
        return Shape(self.id, frozenset(cells), self.color).normalized()

    # ── Set operations ────────────────────────────────────────────────────

    def overlaps(self, other: Shape) -> bool:
        """True if the two blocks share at least one cell."""
        return bool(self.cells & other.cells)

    def merged_with(self, other: Shape) -> Shape:
        """Returns the union of the two blocks (same id and color as self)."""
        return Shape(self.id, self.cells | other.cells, self.color)

    def __repr__(self) -> str:
        return (
            f"Shape(#{self.id}  "
            f"A={self.area}  P={self.perimeter}  c={self.compactness:.3f})"
        )

## RANDOM GENERATION OF GEOMETRIC BLOCKS

In [ ]:
def _neighbors(cell: Cell) -> set[Cell]:
    """Returns the 4 orthogonal neighbors of a cell."""
    x, y = cell
    return {(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)}


def _generate_one(size: int, rng: random.Random) -> set[Cell]:
    """
    Generates a random connected polyomino of `size` cells
    using a random walk on the grid.
    """
    cells: set[Cell] = {(0, 0)}
    frontier: list[Cell] = list(_neighbors((0, 0)))

    while len(cells) < size and frontier:
        cell = rng.choice(frontier)
        cells.add(cell)
        for nb in _neighbors(cell):
            if nb not in cells and nb not in frontier:
                frontier.append(nb)
        frontier = [c for c in frontier if c not in cells]

    return cells


def generate_shapes(
    n: int,
    min_size: int = 3,
    max_size: int = 8,
    seed: Optional[int] = None,
) -> list[Shape]:
    """
    Generates `n` random polyominoes, spaced horizontally for visualization.

    Parameters
    ----------
    n        : number of shapes to generate
    min_size : minimum size of a piece (in cells)
    max_size : maximum size of a piece (in cells)
    seed     : seed for reproducibility (None = random)

    Returns
    -------
    list of Shape, with ids from 0 to n-1
    """
    rng = random.Random(seed)
    palette = [
        "#FF6B6B", "#4ECDC4", "#FFE66D", "#A29BFE", "#FD79A8",
        "#6C5CE7", "#00B894", "#FDCB6E", "#E17055", "#74B9FF",
        "#55EFC4", "#B2BEC3", "#FAB1A0", "#00CEC9", "#D63031",
    ]

    shapes: list[Shape] = []
    offset_x = 0

    for i in range(n):
        size = rng.randint(min_size, max_size)
        raw = _generate_one(size, rng)
        min_x = min(x for x, _ in raw)
        min_y = min(y for _, y in raw)
        cells = frozenset((x - min_x + offset_x, y - min_y) for x, y in raw)
        offset_x = max(x for x, _ in cells) + 3
        shapes.append(Shape(i, cells, palette[i % len(palette)]))

    return shapes

## VERTEX TOPOLOGY: TIPS & NOTCHES ("POINTES" & "CREUX")

In [ ]:
@dataclass
class Vertex:
    """
    Vertex of geometric interest on the contour of a polyomino.

    Attributes
    ----------
    cx, cy   : coordinates of the corner (on the point grid)
    vtype    : 'pointe' (convex tip, 90°) or 'creux' (concave notch, 270°)
    quadrant : quadrant Q0–Q3 in which the associated cell lies
    order    : ω = (d1, d2) lengths of the adjacent boundary segments

    Notes
    -----
    Quadrant convention for a corner (cx, cy):
      Q0 = cell (cx-1, cy-1)    Q1 = cell (cx,   cy-1)
      Q2 = cell (cx-1, cy  )    Q3 = cell (cx,   cy  )
    """

    cx: int
    cy: int
    vtype: str       # 'pointe' | 'creux'
    quadrant: int    # 0, 1, 2 or 3
    order: tuple[int, int]

    def order_product(self) -> int:
        """d1 × d2: local area of influence of the vertex."""
        return self.order[0] * self.order[1]


def _boundary_run(
    cells: frozenset[Cell],
    vx: int,
    vy: int,
    dx: int,
    dy: int,
) -> int:
    """
    Length of the straight boundary segment starting from the corner (vx, vy)
    in the direction (dx, dy), as long as the segment stays on the block boundary.

    Directions:
      dx=±1, dy=0  → horizontal edge (length measured in units)
      dx=0,  dy=±1 → vertical edge
    """
    length = 0
    x, y = vx, vy

    while True:
        nx, ny = x + dx, y + dy
        if dy == 0:
            # Horizontal edge: separating (min(x,nx), y-1) and (min(x,nx), y)
            cx_cell = min(x, nx)
            on_boundary = ((cx_cell, y - 1) in cells) != ((cx_cell, y) in cells)
        else:
            # Vertical edge: separating (x-1, min(y,ny)) and (x, min(y,ny))
            cy_cell = min(y, ny)
            on_boundary = ((x - 1, cy_cell) in cells) != ((x, cy_cell) in cells)

        if not on_boundary:
            break
        length += 1
        x, y = nx, ny

    return length


def get_vertices(shape: Shape) -> list[Vertex]:
    """
    Computes all vertices of interest (tips and notches) of a polyomino.

    Algorithm
    ---------
    For each corner (cx, cy) shared by cells of the block:
      - Count the cells present among the 4 adjacent quadrants.
      - count = 1 → Tip   ("pointe", convex vertex, interior angle 90°)
      - count = 3 → Notch ("creux", concave vertex, interior angle 270°)
      - count ∈ {0, 2, 4} → neutral corner (ignored)

    The order ω = (d1, d2) is computed by `_boundary_run`.
    """
    cells = shape.cells
    has = lambda x, y: (x, y) in cells  # noqa: E731

    # Collect all corners (angles) of the boundary
    corners: set[Cell] = set()
    for x, y in cells:
        for ax in range(2):
            for ay in range(2):
                corners.add((x + ax, y + ay))

    vertices: list[Vertex] = []
    for cx, cy in corners:
        q = [
            1 if has(cx - 1, cy - 1) else 0,  # Q0
            1 if has(cx,     cy - 1) else 0,  # Q1
            1 if has(cx - 1, cy    ) else 0,  # Q2
            1 if has(cx,     cy    ) else 0,  # Q3
        ]
        n_occ = sum(q)

        if n_occ == 1:
            vtype, quadrant = "pointe", q.index(1)
        elif n_occ == 3:
            vtype, quadrant = "creux", q.index(0)
        else:
            continue  # neutral corner: ignored

        # Computation of the order ω = (d_horizontal, d_vertical)
        d_h = (
            _boundary_run(cells, cx, cy, +1, 0)
            + _boundary_run(cells, cx, cy, -1, 0)
        )
        d_v = (
            _boundary_run(cells, cx, cy, 0, +1)
            + _boundary_run(cells, cx, cy, 0, -1)
        )
        vertices.append(
            Vertex(cx, cy, vtype, quadrant, (max(d_h, 1), max(d_v, 1)))
        )

    return vertices

## ELEMENTARY MERGE BY ALIGNMENT

In [ ]:
@dataclass
class FusionCandidate:
    """
    Result of an attempted merge between two blocks.

    Attributes
    ----------
    shape       : block resulting from the merge
    compactness : c(f) of the resulting block
    pointe      : tip vertex used for the alignment (None if merged by surface)
    creux       : notch vertex used for the alignment (None if merged by surface)
    rotation    : rotation applied to f2 (0–3, in multiples of 90°)
    """

    shape: Shape
    compactness: float
    pointe: Optional[Vertex]
    creux: Optional[Vertex]
    rotation: int


def _try_align(
    f1: Shape,
    f2_rot: Shape,
    p: Vertex,
    c: Vertex,
    rotation: int,
) -> Optional[FusionCandidate]:
    """
    Attempts to merge f2_rot into f1 by translating f2_rot so that corner `c`
    coincides with corner `p`. Returns None if an overlap is detected.
    """
    dx, dy = p.cx - c.cx, p.cy - c.cy
    f2_moved = f2_rot.translated(dx, dy)

    if f1.overlaps(f2_moved):
        return None

    merged = f1.merged_with(f2_moved)
    return FusionCandidate(merged, merged.compactness, p, c, rotation)


def all_valid_pairs(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    Enumerates all admissible (tip, notch) pairs between f1 and f2
    over the 4 rotations of f2, without overlap.

    Case 1 : tip   of f1 ↔ notch of f2_rot (same quadrant)
    Case 2 : notch of f1 ↔ tip   of f2_rot (same quadrant)
    """
    candidates: list[FusionCandidate] = []
    v1 = get_vertices(f1)
    pointes1 = [v for v in v1 if v.vtype == "pointe"]
    creux1   = [v for v in v1 if v.vtype == "creux"]

    for r in range(4):
        f2r = f2.rotated(r)
        v2 = get_vertices(f2r)
        pointes2 = [v for v in v2 if v.vtype == "pointe"]
        creux2   = [v for v in v2 if v.vtype == "creux"]

        for p in pointes1:
            for c in creux2:
                if p.quadrant == c.quadrant:
                    result = _try_align(f1, f2r, p, c, r)
                    if result:
                        candidates.append(result)

        for c in creux1:
            for p in pointes2:
                if p.quadrant == c.quadrant:
                    result = _try_align(f1, f2r, c, p, r)
                    if result:
                        candidates.append(result)

    return candidates

## MERGE BY CLOSEST SURFACE LENGTH

In [ ]:
def _contact_edges(f1: Shape, f2_moved: Shape) -> int:
    """Counts the contact edges between f1 and f2_moved (adjacency without overlap)."""
    count = 0
    for x, y in f1.cells:
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            if (x + dx, y + dy) in f2_moved.cells:
                count += 1
    return count


def _h_runs(shape: Shape) -> list[tuple[int, int, int]]:
    """Returns the horizontal edge runs: (y_corner, x_start, length)."""
    cells, runs = shape.cells, []
    for y in sorted({y for _, y in cells}):
        for dy in [0, 1]:
            y_corner = y + dy
            row = sorted(x for x, yy in cells if yy == y)
            if not row:
                continue
            start, length = row[0], 1
            for i in range(1, len(row)):
                if row[i] == row[i - 1] + 1:
                    length += 1
                else:
                    runs.append((y_corner, start, length))
                    start, length = row[i], 1
            runs.append((y_corner, start, length))
    return runs


def _v_runs(shape: Shape) -> list[tuple[int, int, int]]:
    """Returns the vertical edge runs: (x_corner, y_start, length)."""
    cells, runs = shape.cells, []
    for x in sorted({x for x, _ in cells}):
        for dx in [0, 1]:
            x_corner = x + dx
            col = sorted(y for xx, y in cells if xx == x)
            if not col:
                continue
            start, length = col[0], 1
            for i in range(1, len(col)):
                if col[i] == col[i - 1] + 1:
                    length += 1
                else:
                    runs.append((x_corner, start, length))
                    start, length = col[i], 1
            runs.append((x_corner, start, length))
    return runs


def merge_by_surface(f1: Shape, f2: Shape) -> Optional[FusionCandidate]:
    """
    Used when there is no notch in either f1 or f2.
    Aligns the boundary runs of closest lengths and maximizes
    the number of contact edges.
    """
    best_contact = -1
    best: Optional[FusionCandidate] = None
    h_runs1, v_runs1 = _h_runs(f1), _v_runs(f1)

    for r in range(4):
        f2r = f2.rotated(r)

        for yc1, xs1, l1 in h_runs1:
            for yc2, xs2, l2 in _h_runs(f2r):
                if abs(l1 - l2) > max(l1, l2):
                    continue
                dx = xs1 - xs2
                for adj_dy in [yc1 - yc2 - 1, yc1 - yc2]:
                    f2m = f2r.translated(dx, adj_dy)
                    if f1.overlaps(f2m):
                        continue
                    c = _contact_edges(f1, f2m)
                    if c > best_contact:
                        best_contact = c
                        mg = f1.merged_with(f2m)
                        best = FusionCandidate(mg, mg.compactness, None, None, r)

        for xc1, ys1, l1 in v_runs1:
            for xc2, ys2, l2 in _v_runs(f2r):
                if abs(l1 - l2) > max(l1, l2):
                    continue
                dy = ys1 - ys2
                for adj_dx in [xc1 - xc2 - 1, xc1 - xc2]:
                    f2m = f2r.translated(adj_dx, dy)
                    if f1.overlaps(f2m):
                        continue
                    c = _contact_edges(f1, f2m)
                    if c > best_contact:
                        best_contact = c
                        mg = f1.merged_with(f2m)
                        best = FusionCandidate(mg, mg.compactness, None, None, r)

    return best


def _best(candidates: list[FusionCandidate]) -> Optional[FusionCandidate]:
    """Returns the candidate with maximum compactness, or None if the list is empty."""
    return max(candidates, key=lambda c: c.compactness) if candidates else None

## MERGE SELECTION HEURISTICS

### HEURISTIC 1 - Maximum order

In [ ]:
def heuristic_1_candidates(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    H1 — Selection by maximum order.

    For each rotation of f2:
      Case A : max-order tip   of f1 (d1×d2) ↔ max-order notch of f2 in the same quadrant
      Case B : max-order notch of f1          ↔ max-order tip   of f2 in the same quadrant

    Only one pair is tested per case/rotation → complexity O(V).

    Key fix: filtering by quadrant guarantees geometric compatibility.
    Without this filtering, the maximum-order vertices may lie in opposite quadrants,
    causing a systematic overlap.
    """

    def argmax_order(verts: list[Vertex]) -> Optional[Vertex]:
        return max(verts, key=lambda v: v.order_product(), default=None)

    def argmax_compatible(verts: list[Vertex], quadrant: int) -> Optional[Vertex]:
        """Among the vertices of quadrant `quadrant`, returns the one with max order."""
        compatible = [v for v in verts if v.quadrant == quadrant]
        return argmax_order(compatible)

    candidates: list[FusionCandidate] = []
    v1 = get_vertices(f1)
    pointes1 = [v for v in v1 if v.vtype == "pointe"]
    creux1   = [v for v in v1 if v.vtype == "creux"]

    for r in range(4):
        f2r = f2.rotated(r)
        v2 = get_vertices(f2r)
        pointes2 = [v for v in v2 if v.vtype == "pointe"]
        creux2   = [v for v in v2 if v.vtype == "creux"]

        # Case A: max tip of f1 → max compatible notch of f2
        p_max = argmax_order(pointes1)
        if p_max:
            c_compat = argmax_compatible(creux2, p_max.quadrant)
            if c_compat:
                result = _try_align(f1, f2r, p_max, c_compat, r)
                if result:
                    candidates.append(result)

        # Case B: max notch of f1 → max compatible tip of f2
        c_max = argmax_order(creux1)
        if c_max:
            p_compat = argmax_compatible(pointes2, c_max.quadrant)
            if p_compat:
                result = _try_align(f1, f2r, c_max, p_compat, r)
                if result:
                    candidates.append(result)

    return candidates

### HEURISTIC 2 - Order-product distance

In [ ]:
def heuristic_2_candidates(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    H2 — Selection by minimizing the dimensional gap and maximizing
    the contact surface.

    Step 1 — Building the set I:
      I = { i | Δω[0] = min_j Δω[0]  OR  Δω[1] = min_j Δω[1] }
      (pairs whose orders are the closest dimensionally)

    Step 2 — Refinement to I':
      I' = { i ∈ I | prod[0] = max prod[0]  OR  prod[1] = max prod[1] }
      (among I, the pairs with the largest area of influence)

    Complexity: O(V²) — explores all valid pairs before filtering.

    Automatic fallback to `merge_by_surface` if no notch exists.
    """
    candidates = all_valid_pairs(f1, f2)
    if not candidates:
        result = merge_by_surface(f1, f2)
        return [result] if result else []

    def delta(fc: FusionCandidate) -> tuple[int, int]:
        if fc.pointe is None or fc.creux is None:
            return (0, 0)
        return (
            abs(fc.pointe.order[0] - fc.creux.order[0]),
            abs(fc.pointe.order[1] - fc.creux.order[1]),
        )

    def product(fc: FusionCandidate) -> tuple[int, int]:
        if fc.pointe is None or fc.creux is None:
            return (0, 0)
        return (
            fc.pointe.order[0] * fc.creux.order[0],
            fc.pointe.order[1] * fc.creux.order[1],
        )

    # Building I
    deltas = [delta(c) for c in candidates]
    min_d0 = min(d[0] for d in deltas)
    min_d1 = min(d[1] for d in deltas)
    I = (
        [c for c, d in zip(candidates, deltas) if d[0] == min_d0 or d[1] == min_d1]
        or candidates
    )

    # Refinement to I'
    products = [product(c) for c in I]
    max_p0   = max(p[0] for p in products)
    max_p1   = max(p[1] for p in products)
    I_prime  = (
        [c for c, p in zip(I, products) if p[0] == max_p0 or p[1] == max_p1]
        or I
    )

    return I_prime

### HEURISTIC 3 - Greedy brute force

In [ ]:
def heuristic_3_candidates(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    H3 — Greedy brute force: evaluates all valid (tip, notch) pairs
    over the 4 rotations of f2 without any filtering.

    Serves as the absolute baseline to evaluate H1 and H2.
    Complexity: O(V1 × V2 × 4).

    Automatic fallback to `merge_by_surface` if no notch exists.
    """
    v = get_vertices(f1) + get_vertices(f2)

    if not any(vertex.vtype == "creux" for vertex in v):
        result = merge_by_surface(f1, f2)
        return [result] if result else []

    candidates = all_valid_pairs(f1, f2)
    if candidates:
        return candidates

    # Fallback: no valid tip-notch pair → merge by surface
    result = merge_by_surface(f1, f2)
    return [result] if result else []

## MULTI-BRANCH GREEDY ALGORITHM

In [ ]:
@dataclass
class StepLog:
    """
    Record of one step of the greedy algorithm.

    Attributes
    ----------
    step      : step number (1-indexed)
    step_type : 'fusion' | 'local_optimum' | 'done'
    f1_id     : identifier of the selected most compact block
    f2_id     : identifier of the second selected block
    c_before  : compactness of f1 and f2 before the merge
    c_after   : compactness of the resulting block (0.0 if local optimum)
    message   : textual description of the step
    """

    step: int
    step_type: str
    f1_id: int
    f2_id: int
    c_before: tuple[float, float]
    c_after: float
    message: str


def _state_key(shapes: list[Shape]) -> frozenset:
    """Unique state key: set of the cells of each shape (order-independent)."""
    return frozenset(s.cells for s in shapes)


# ── Global beam search parameter ─────────────────────────────────────────
#   0 → unlimited (exact search, beware of memory for n > 8)
#   N → keep the N best active branches at each step
MAX_BRANCHES: int = 150

# ── Dispatch tables ───────────────────────────────────────────────────────
_HEURISTIC_FN: dict = {
    1: heuristic_1_candidates,
    2: heuristic_2_candidates,
    3: heuristic_3_candidates,
}

_HEURISTIC_LABEL: dict[int, str] = {
    1: "H1 — Maximum order",
    2: "H2 — Order-product distance",
    3: "H3 — Greedy brute force",
}


def greedy_compact_assembly(
    shapes: list[Shape],
    heuristic: int = 3,
    verbose: bool = True,
) -> tuple[list[Shape], list[StepLog], list[list[StepLog]], list[Shape]]:
    """
    Multi-branch greedy algorithm with beam search.

    At each step:
      1. The 2 shapes of maximum compactness are selected (f1, f2).
      2. The `heuristic` generates N merge candidates.
      3. Each candidate opens an independent branch.
      4. The beam search prunes the branches exceeding MAX_BRANCHES.

    Identical states (same set of cells) are deduplicated.

    Parameters
    ----------
    shapes    : list of blocks to assemble
    heuristic : 1 → H1, 2 → H2, 3 → H3
    verbose   : print progress to the console

    Returns
    -------
    best_shapes  : final shapes of the best branch (max compactness)
    best_logs    : log of that branch
    all_branches : logs of all branches (for visualization)
    best_trace   : list of intermediate merged shapes (best branch)
    """
    sep = "=" * 65
    label = _HEURISTIC_LABEL.get(heuristic, str(heuristic))
    fn    = _HEURISTIC_FN.get(heuristic)
    if fn is None:
        raise ValueError(f"Unknown heuristic: {heuristic}. Valid values: 1, 2, 3.")

    if verbose:
        print(sep)
        print(f"  MULTI-BRANCH GREEDY — {label}")
        print(
            f"  {len(shapes)} shapes  |  MAX_BRANCHES="
            f"{'∞' if MAX_BRANCHES == 0 else MAX_BRANCHES}"
        )
        print(sep)

    # State: StateKey → (current_shapes, logs, trace_of_merged_shapes)
    active: dict = {_state_key(shapes): (list(shapes), [], [])}
    step = 0

    def _is_stuck(logs: list[StepLog]) -> bool:
        """True if the last log entry is a local optimum."""
        return bool(logs) and logs[-1].step_type == "local_optimum"

    def _score(item: tuple) -> float:
        """Branch score = maximum compactness among its current shapes."""
        return max(s.compactness for s in item[0])

    def _beam_trim(d: dict) -> dict:
        """
        Prunes the dictionary of active branches to keep only MAX_BRANCHES.
        Finished branches (len == 1) or stuck ones are always kept.
        """
        if MAX_BRANCHES == 0 or len(d) <= MAX_BRANCHES:
            return d
        done = {k: v for k, v in d.items() if len(v[0]) <= 1 or _is_stuck(v[1])}
        actv = {k: v for k, v in d.items() if k not in done}
        sorted_active = sorted(actv.items(), key=lambda kv: _score(kv[1]), reverse=True)
        quota = max(1, MAX_BRANCHES - len(done))
        return dict(list(done.items()) + sorted_active[:quota])

    def _get_candidates(fi: Shape, fj: Shape) -> list[FusionCandidate]:
        """Calls the heuristic, with fallback to merge_by_surface if there is no notch."""
        v = get_vertices(fi) + get_vertices(fj)
        if not any(x.vtype == "creux" for x in v):
            result = merge_by_surface(fi, fj)
            return [result] if result else []
        return fn(fi, fj)

    # ── Main loop ─────────────────────────────────────────────────────────
    while True:
        to_merge = {
            k: v for k, v in active.items()
            if len(v[0]) > 1 and not _is_stuck(v[1])
        }
        if not to_merge:
            break

        step += 1
        new_active = {
            k: v for k, v in active.items()
            if len(v[0]) <= 1 or _is_stuck(v[1])
        }

        for key, (current, logs, trace) in to_merge.items():
            # Selection of the two most compact blocks
            sorted_shapes = sorted(current, key=lambda x: x.compactness, reverse=True)
            f1, f2, rest = sorted_shapes[0], sorted_shapes[1], sorted_shapes[2:]
            c1, c2 = f1.compactness, f2.compactness
            candidates = _get_candidates(f1, f2)

            if not candidates:
                log = StepLog(
                    step, "local_optimum", f1.id, f2.id, (c1, c2), 0.0,
                    f"  [{step:>3}] Local optimum — {len(current)} unmerged shape(s)",
                )
                if key not in new_active:
                    new_active[key] = (current, logs + [log], trace)
                continue

            for cand in candidates:
                fused = Shape(f1.id, cand.shape.cells, f1.color)
                new_shapes = [fused] + list(rest)
                new_key    = _state_key(new_shapes)
                log = StepLog(
                    step, "fusion", f1.id, f2.id, (c1, c2),
                    fused.compactness,
                    f"  [{step:>3}] #{f1.id} + #{f2.id}"
                    f" → c={fused.compactness:.4f}  [{len(new_shapes)} remaining]",
                )
                new_logs  = logs + [log]
                new_trace = trace + [fused]

                if new_key not in new_active:
                    new_active[new_key] = (new_shapes, new_logs, new_trace)
                else:
                    # Deduplication: keep the most compact branch
                    existing_c = max(x.compactness for x in new_active[new_key][0])
                    if fused.compactness > existing_c:
                        new_active[new_key] = (new_shapes, new_logs, new_trace)

        active = _beam_trim(new_active)

        if verbose:
            n_done = sum(1 for v in active.values() if len(v[0]) == 1)
            print(f"  Step {step}: {len(active)} branch(es)  ({n_done} completed)")

    # ── Finalization ──────────────────────────────────────────────────────
    final: dict = {}
    for key, (sh, lg, tr) in active.items():
        if len(sh) == 1:
            s = sh[0]
            lg = lg + [StepLog(
                step + 1, "done", -1, -1, (0.0, 0.0), s.compactness,
                f"       ✔  c={s.compactness:.4f}  A={s.area}  P={s.perimeter}",
            )]
        final[key] = (sh, lg, tr)

    def _score_item(item: tuple) -> float:
        return max(s.compactness for s in item[0])

    best_shapes, best_logs, best_trace = max(final.values(), key=_score_item)
    c_best = max(s.compactness for s in best_shapes)

    if verbose:
        print(sep)
        print(f"  {len(final)} branch(es)  |  best c = {c_best:.4f}")
        if len(best_shapes) == 1:
            s = best_shapes[0]
            print(f"  ✔  Complete assembly  A={s.area}  P={s.perimeter}")
        else:
            print(f"  ⚠  {len(best_shapes)} unmerged shape(s)")
        print(sep)

    all_branches = [v[1] for v in final.values()]
    return best_shapes, best_logs, all_branches, best_trace

## MORPHOLOGICAL RESEMBLANCE METRIC

In [ ]:
def _align_shape(reference: Shape, candidate: Shape) -> frozenset[Cell]:
    """
    Searches for the rotation r ∈ {0°, 90°, 180°, 270°} and the translation (dx, dy)
    that minimize the cardinality of the symmetric difference between `reference`
    and the aligned `candidate`.

    Strategy: for each rotation, test all translations that
    superimpose at least one cell of `candidate` onto a cell of `reference`.

    Returns the cells of `candidate` after the optimal transformation.
    """
    ref_cells  = reference.cells
    best_diff  = len(ref_cells) + len(candidate.cells) + 1  # worst case + 1
    best_cells = candidate.cells

    for r in range(4):
        rot_cells = candidate.rotated(r).normalized().cells
        for rx, ry in ref_cells:
            for cx, cy in rot_cells:
                dx, dy = rx - cx, ry - cy
                moved = frozenset((x + dx, y + dy) for x, y in rot_cells)
                sym_diff = len(ref_cells.symmetric_difference(moved))
                if sym_diff < best_diff:
                    best_diff  = sym_diff
                    best_cells = moved

    return best_cells


def resemblance_simple(shape_a: Shape, shape_b: Shape) -> float:
    """
    Simple (unweighted) resemblance metric.

    R = N_diff / A_tot = |A Δ B| / |A ∪ B|

    where A Δ B is the symmetric difference after optimal alignment.
    R ∈ [0, 1]: 0 → identical shapes, 1 → no common cell.
    """
    aligned_b = _align_shape(shape_a, Shape(shape_b.id, shape_b.cells))
    union      = shape_a.cells | aligned_b
    sym_diff   = shape_a.cells.symmetric_difference(aligned_b)
    a_tot      = len(union)
    return len(sym_diff) / a_tot if a_tot > 0 else 0.0


def _cell_dist_to_shape(cell: Cell, shape_cells: frozenset[Cell]) -> int:
    """
    Minimum Manhattan distance between `cell` and the set `shape_cells`.
    Used as the eccentricity degree w_i in the weighted metric.
    """
    cx, cy = cell
    return min(abs(cx - sx) + abs(cy - sy) for sx, sy in shape_cells)


def resemblance_weighted(shape_a: Shape, shape_b: Shape) -> float:
    """
    Weighted resemblance metric with eccentricity degree.

    R_w = Σ w_i / A_tot

    where w_i = d_Manhattan(cell_i, perimeter of shape_a) + 1
      (+1 guarantees w_i ≥ 1 even for cells adjacent to the perimeter)

    Divergent cells far from the reference perimeter are penalized
    more heavily than nearby cells, reflecting a more severe
    morphological divergence.
    """
    aligned_b = _align_shape(shape_a, Shape(shape_b.id, shape_b.cells))
    union      = shape_a.cells | aligned_b
    sym_diff   = shape_a.cells.symmetric_difference(aligned_b)
    a_tot      = len(union)

    if a_tot == 0:
        return 0.0

    weighted_sum = sum(
        _cell_dist_to_shape(cell, shape_a.cells) + 1
        for cell in sym_diff
    )
    return weighted_sum / a_tot


def compute_resemblance(
    results: dict[int, tuple],
) -> dict[int, dict[str, float]]:
    """
    Computes R_simple and R_weighted for H1 and H2 relative to H3 (fixed reference).

    H3 is excluded from the result (reference = 0 by definition).

    Parameters
    ----------
    results : dict { heuristic_id: (best_shapes, logs, branches, trace) }

    Returns
    -------
    dict { heuristic_id: {"simple": R_s, "weighted": R_w} }
    """
    ref_shapes = results[3][0]
    ref_shape  = Shape(-1, frozenset(c for s in ref_shapes for c in s.cells))

    metrics: dict[int, dict[str, float]] = {}
    for hi, (best, *_) in results.items():
        if hi == 3:
            continue
        cand_shape = Shape(-1, frozenset(c for s in best for c in s.cells))
        metrics[hi] = {
            "simple":   resemblance_simple(ref_shape, cand_shape),
            "weighted": resemblance_weighted(ref_shape, cand_shape),
        }
    return metrics

## VISUALIZATION

### color palette

In [ ]:
BG     = "white"
PANEL  = "#f8f9fa"
BORDER = "#ced4da"
TEXT   = "#212529"
MUTED  = "#6c757d"

C_H1 = "#e67700"  # orange   — H1
C_H2 = "#1971c2"  # blue     — H2
C_H3 = "#7048e8"  # purple   — H3

_COLOR  = {1: C_H1, 2: C_H2, 3: C_H3}
_LS     = {1: "--",  2: "-",   3: "-."}
_MARKER = {1: "o",   2: "o",   3: "D"}
_LABEL  = {
    1: "H1 — Maximum order",
    2: "H2 — Order-product distance",
    3: "H3 — Greedy brute force",
}

### STYLE HELPERS

In [ ]:
def _style_ax(ax: plt.Axes) -> None:
    """Applies the common style (background, grid, borders)."""
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
        sp.set_linewidth(0.8)
    ax.tick_params(colors=TEXT, labelsize=8)
    ax.grid(True, color=BORDER, linewidth=0.5, alpha=0.9, zorder=0)


def _plot_branches(
    ax: plt.Axes,
    all_branches: list[list[StepLog]],
    c_init: float,
    color: str,
    alpha: float = 0.13,
    lw: float = 0.6,
) -> None:
    """Plots all branches as a watermark (thin, semi-transparent lines)."""
    for logs in all_branches:
        fusions = [l for l in logs if l.step_type == "fusion"]
        if not fusions:
            continue
        xs = list(range(len(fusions) + 1))
        ys = [c_init] + [l.c_after for l in fusions]
        ax.plot(xs, ys, color=color, linewidth=lw, alpha=alpha, linestyle="-", zorder=2)


def _plot_best(
    ax: plt.Axes,
    logs: list[StepLog],
    c_init: float,
    color: str,
    label: str,
    linestyle: str = "-",
    marker: str = "o",
    lw: float = 2.3,
    ms: float = 9,
) -> None:
    """Plots the best branch in bold, with an annotation of the final value."""
    fusions = [l for l in logs if l.step_type == "fusion"]
    if not fusions:
        return
    xs = list(range(len(fusions) + 1))
    ys = [c_init] + [l.c_after for l in fusions]
    ax.plot(
        xs, ys,
        linestyle=linestyle, marker=marker, color=color,
        linewidth=lw, markersize=ms,
        markerfacecolor="white", markeredgecolor=color, markeredgewidth=2.0,
        label=label, zorder=6,
    )
    ax.annotate(
        f"  {ys[-1]:.4f}", xy=(xs[-1], ys[-1]),
        fontsize=8.5, color=color, fontfamily="monospace", va="center", zorder=7,
    )


def _finalise_evo_ax(ax: plt.Axes, c_init: float, title: str) -> None:
    """Adds the legend, labels and title to an evolution axis."""
    ax.axhline(c_init, color=MUTED, linewidth=1.0, linestyle=":",
               label=f"initial c̄ = {c_init:.3f}", zorder=1)
    ax.set_xlabel("Merge step", fontsize=10, color=TEXT, labelpad=7)
    ax.set_ylabel("c(f) = Area / Perimeter", fontsize=10, color=TEXT, labelpad=7)
    ax.legend(fontsize=9, facecolor="white", edgecolor=BORDER,
              labelcolor=TEXT, framealpha=1.0, loc="upper left", frameon=True)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.set_title(title, color=TEXT, fontsize=11, fontweight="bold", pad=10, loc="left")


def _fig_header(fig: plt.Figure, title: str, subtitle: str) -> None:
    """Adds a centered title and subtitle at the top of the figure."""
    fig.suptitle(title, color=TEXT, fontsize=13, fontweight="bold", y=0.97)
    fig.text(0.5, 0.925, subtitle, ha="center", color=MUTED, fontsize=8.5)


def _draw_cells(
    ax: plt.Axes,
    shapes: list[Shape],
    color: str,
    label: Optional[str] = None,
    lw_border: float = 1.8,
) -> None:
    """Draws the cells of a list of shapes, centered in the axis."""
    all_cells = [(x, y) for s in shapes for x, y in s.cells]
    if not all_cells:
        return

    min_x = min(x for x, _ in all_cells)
    min_y = min(y for _, y in all_cells)
    centered = [(x - min_x, y - min_y) for x, y in all_cells]

    face = to_rgba(color, alpha=0.60)
    edge = to_rgba(color, alpha=1.00)
    for x, y in centered:
        ax.add_patch(mpatches.FancyBboxPatch(
            (x + 0.06, y + 0.06), 0.88, 0.88,
            boxstyle="round,pad=0.04",
            facecolor=face, edgecolor=edge, linewidth=1.2, zorder=3,
        ))

    xs  = [x for x, _ in centered]
    ys  = [y for _, y in centered]
    pad = 1.5
    ax.set_xlim(min(xs) - pad, max(xs) + 1 + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + 1 + pad)
    ax.set_aspect("equal")
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(color)
        sp.set_linewidth(lw_border)
    ax.grid(True, color=BORDER, linewidth=0.4, alpha=0.8, zorder=0)
    ax.tick_params(labelbottom=False, labelleft=False, bottom=False, left=False)

    if label:
        ax.set_title(label, color=color, fontsize=10, fontweight="bold", pad=6)


def _shape_info(
    shapes: list[Shape],
    logs: list[StepLog],
) -> tuple[int, bool, int, int, float]:
    """Returns (n_merges, complete, area, perimeter, compactness) for a result."""
    n_fusions = sum(1 for l in logs if l.step_type == "fusion")
    success   = len(shapes) == 1
    merged    = Shape(-1, frozenset(c for s in shapes for c in s.cells))
    a, p      = merged.area, merged.perimeter
    return n_fusions, success, a, p, (a / p if p else 0.0)

### EVOLUTION PLOTS

In [ ]:
def _save_evolution_global(
    initial: list[Shape],
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """Global plot: one curve per heuristic (best branch only)."""
    fig, ax = plt.subplots(figsize=(13, 7), facecolor=BG)
    fig.subplots_adjust(left=0.09, right=0.96, top=0.87, bottom=0.12)
    _style_ax(ax)
    c_init = sum(s.compactness for s in initial) / len(initial)

    for hi in sorted(results):
        _, logs, _, _ = results[hi]
        _plot_best(ax, logs, c_init, _COLOR[hi], _LABEL[hi], _LS[hi], _MARKER[hi])

    _finalise_evo_ax(ax, c_init, "H1 / H2 / H3 comparison — best compactness reached")
    _fig_header(
        fig,
        "Compactness evolution across merges",
        f"{len(initial)} shapes  ·  seed={CONFIG.get('seed', '—')}  ·  "
        f"size {CONFIG.get('min_size')}–{CONFIG.get('max_size')}",
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [global evolution]       → {filepath}")


def _save_comparaison(
    initial: list[Shape],
    hi_a: int,
    hi_b: int,
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """Watermarked branches + best curve for two heuristics."""
    fig, ax = plt.subplots(figsize=(11, 6), facecolor=BG)
    fig.subplots_adjust(left=0.10, right=0.97, top=0.87, bottom=0.13)
    _style_ax(ax)
    c_init = sum(s.compactness for s in initial) / len(initial)

    for hi in (hi_a, hi_b):
        _, logs, branches, _ = results[hi]
        _plot_branches(ax, branches, c_init, _COLOR[hi])
        _plot_best(
            ax, logs, c_init, _COLOR[hi],
            _LABEL[hi] + " (best branch)",
            _LS[hi], _MARKER[hi], lw=2.5, ms=10,
        )

    ax.plot([], [], color=MUTED, linewidth=0.7, alpha=0.4, label="Explored branches")
    lbl_a = _LABEL[hi_a].split("—")[0].strip()
    lbl_b = _LABEL[hi_b].split("—")[0].strip()
    _finalise_evo_ax(ax, c_init, f"{lbl_a}  vs  {lbl_b}  —  explored branches")
    _fig_header(
        fig, f"Comparison {lbl_a} / {lbl_b}",
        f"{len(initial)} shapes  ·  seed={CONFIG.get('seed', '—')}",
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [comparison {lbl_a}/{lbl_b}]   → {filepath}")

### SHAPE PLOTS

In [ ]:
def _save_all_shapes(
    initial: list[Shape],
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """Displays the final shapes of all heuristics side by side."""
    n_panels = len(results)
    fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels + 2, 7), facecolor=BG)
    if n_panels == 1:
        axes = [axes]
    fig.subplots_adjust(left=0.01, right=0.99, top=0.82, bottom=0.16, wspace=0.28)

    short_labels = {1: "H1\nMaximum order", 2: "H2\nOrder-product", 3: "H3\nBrute force"}

    for ax, hi in zip(axes, sorted(results)):
        best, logs, _, _ = results[hi]
        _draw_cells(ax, best, _COLOR[hi], label=short_labels.get(hi, f"H{hi}"))
        n, ok, a, p, c = _shape_info(best, logs)
        status = "✔ complete" if ok else f"⚠ {len(best)} fragment(s)"
        ax.text(
            0.5, -0.10,
            f"A={a}  P={p}  c={c:.4f}\n{status}  ({n} merges)",
            transform=ax.transAxes, ha="center", va="top",
            fontsize=7.5, color=MUTED, fontfamily="monospace",
        )

    fig.suptitle(
        "Final shapes — H1 / H2 / H3 comparison",
        color=TEXT, fontsize=13, fontweight="bold", y=0.97,
    )
    fig.text(
        0.5, 0.905,
        f"seed={CONFIG.get('seed', '—')}  ·  "
        f"size {CONFIG.get('min_size')}–{CONFIG.get('max_size')} cells",
        ha="center", color=MUTED, fontsize=8.5,
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [final shapes]           → {filepath}")

### RESEMBLANCE PLOTS

In [ ]:
def _save_resemblance(
    results: dict[int, tuple],
    metrics: dict[int, dict[str, float]],
    filepath: str,
) -> None:
    """Bar chart: R_simple and R_weighted of H1 and H2 relative to H3."""
    heuristics = sorted(metrics.keys())
    labels     = [_LABEL[hi].split("—")[0].strip() for hi in heuristics]
    colors     = [_COLOR[hi] for hi in heuristics]
    r_simple   = [metrics[hi]["simple"]   for hi in heuristics]
    r_weighted = [metrics[hi]["weighted"] for hi in heuristics]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), facecolor=BG)
    fig.subplots_adjust(left=0.07, right=0.97, top=0.83, bottom=0.14, wspace=0.32)

    panels = [
        (r_simple,   "① Simple resemblance",                    r"$R = N_{diff}\,/\,A_{tot}$"),
        (r_weighted, "② Weighted resemblance (eccentricity)",   r"$R = \sum w_i\,/\,A_{tot}$"),
    ]

    for ax, (values, title, formula) in zip(axes, panels):
        ax.set_facecolor(PANEL)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER)
            sp.set_linewidth(0.8)
        ax.tick_params(colors=TEXT, labelsize=9)
        ax.grid(True, color=BORDER, linewidth=0.5, alpha=0.9, axis="y", zorder=0)

        bars = ax.bar(labels, values, color=colors, alpha=0.78,
                      edgecolor=colors, linewidth=1.2, zorder=3, width=0.45)

        for bar, v in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values, default=0) * 0.02 + 1e-6,
                f"{v:.4f}", ha="center", va="bottom",
                fontsize=9.5, color=TEXT, fontfamily="monospace", fontweight="bold",
            )

        ax.set_ylim(0, max(values, default=0.1) * 1.22 + 1e-6)
        ax.set_ylabel("R  (0 = identical to H3)", fontsize=9.5, color=TEXT, labelpad=6)
        ax.set_title(f"{title}\n{formula}", color=TEXT, fontsize=10.5,
                     fontweight="bold", pad=8)
        ax.tick_params(axis="x", colors=TEXT, labelsize=10)

    fig.suptitle(
        "Resemblance metric  —  H1 and H2 compared to H3 (reference)",
        color=TEXT, fontsize=12, fontweight="bold", y=0.97,
    )
    fig.text(
        0.5, 0.905,
        "Low R → shape close to H3  ·  "
        "w_i = Manhattan distance from the divergent cell to the perimeter of H3",
        ha="center", color=MUTED, fontsize=8.0,
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [final resemblance]      → {filepath}")


def _save_resemblance_evolution(
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """
    Evolution of the resemblance metrics (simple and weighted) at each merge
    step, for H1 and H2 relative to the final shape of H3.

    Each point corresponds to the intermediate shape accumulated after k merges
    on the best branch of the heuristic concerned.
    A decreasing R indicates morphological convergence toward H3.
    """
    ref_shapes = results[3][0]
    ref_shape  = Shape(-1, frozenset(c for s in ref_shapes for c in s.cells))

    # Computation of the time series
    series: dict[int, dict[str, list[float]]] = {}
    for hi in (1, 2):
        _, _, _, trace = results[hi]
        series[hi] = {
            "simple":   [resemblance_simple(ref_shape, inter)   for inter in trace],
            "weighted": [resemblance_weighted(ref_shape, inter) for inter in trace],
        }

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)
    fig.subplots_adjust(left=0.08, right=0.97, top=0.85, bottom=0.12, wspace=0.30)

    panels = [
        ("simple",   "① Simple resemblance    R = N_diff / A_tot"),
        ("weighted", "② Weighted resemblance  R = Σ w_i / A_tot"),
    ]

    for ax, (metric, title) in zip(axes, panels):
        ax.set_facecolor(PANEL)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER)
            sp.set_linewidth(0.8)
        ax.tick_params(colors=TEXT, labelsize=9)
        ax.grid(True, color=BORDER, linewidth=0.5, alpha=0.9, zorder=0)
        ax.set_xlabel("Merge step", fontsize=10, color=TEXT, labelpad=7)
        ax.set_ylabel("R  (0 = identical to H3)", fontsize=10, color=TEXT, labelpad=7)
        ax.set_title(title, color=TEXT, fontsize=10.5, fontweight="bold", pad=8, loc="left")

        for hi in (1, 2):
            vals = series[hi][metric]
            if not vals:
                continue
            xs = list(range(1, len(vals) + 1))
            ax.plot(
                xs, vals,
                color=_COLOR[hi], linewidth=2.2,
                linestyle=_LS[hi], marker=_MARKER[hi],
                markersize=7, markerfacecolor="white",
                markeredgecolor=_COLOR[hi], markeredgewidth=2.0,
                label=_LABEL[hi], zorder=5,
            )
            ax.annotate(
                f"  {vals[-1]:.4f}", xy=(xs[-1], vals[-1]),
                fontsize=8.5, color=_COLOR[hi], fontfamily="monospace", va="center",
            )

        ax.legend(fontsize=9, facecolor="white", edgecolor=BORDER,
                  labelcolor=TEXT, framealpha=1.0, loc="upper right")

    fig.suptitle(
        "Evolution of morphological resemblance  —  H1 and H2 vs H3",
        color=TEXT, fontsize=12, fontweight="bold", y=0.97,
    )
    fig.text(
        0.5, 0.91,
        "Each point = intermediate accumulated shape of the best branch"
        "  ·  decreasing R → convergence toward H3",
        ha="center", color=MUTED, fontsize=8.0,
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [resemblance evolution]  → {filepath}")


def visualize(
    initial: list[Shape],
    results: dict[int, tuple],
    save_path: str = "compacite",
) -> None:
    """
    Generates 4 images from the results of the 3 heuristics:
      {base}_evolution.png        — H1 / H2 / H3 evolution curves
      {base}_comparaison_h2.png   — H2 vs H3 with explored branches
      {base}_formes.png           — final shapes side by side
    """
    base = save_path.replace(".png", "")
    print()
    _save_evolution_global(initial, results, base + "_evolution.png")
    _save_comparaison(initial, 2, 3, results, base + "_comparaison_h2.png")
    _save_all_shapes(initial, results, base + "_formes.png")

## CONFIGURATION AND ENTRY POINT

In [ ]:
CONFIG: dict = {
    # ── Main instance ───────────────────────────────────────────────────
    "n_shapes" : 10,   # number of polyominoes to generate
    "min_size" : 3,     # minimum size of a piece (cells)
    "max_size" : 8,     # maximum size of a piece (cells)
    "seed"     : 42,    # random seed (None = not reproducible)
    "save"     : "compacite",  # prefix of the generated image files
}


def main(
    n_shapes: int           = CONFIG["n_shapes"],
    min_size: int           = CONFIG["min_size"],
    max_size: int           = CONFIG["max_size"],
    seed:     Optional[int] = CONFIG["seed"],
    save:     str           = CONFIG["save"],
) -> None:
    """
    Main entry point. Compatible with terminal scripts, Jupyter, Kaggle and Colab.

    Runs the 3 heuristics on a random instance and produces 5 images:
      {save}_evolution.png              — H1 / H2 / H3 evolution curves
      {save}_comparaison_h2.png         — H2 vs H3 with branches
      {save}_formes.png                 — final shapes side by side
      {save}_ressemblance.png           — final resemblance metrics
      {save}_ressemblance_evolution.png — temporal evolution of the metrics

    Parameters
    ----------
    n_shapes : number of polyominoes to generate
    min_size : minimum size of a piece (in cells)
    max_size : maximum size of a piece (in cells)
    seed     : random seed for reproducibility
    save     : prefix of the image file names

    Examples
    --------
    >>> main()                               # default parameters
    >>> main(n_shapes=50, seed=7)            # 50 shapes, seed 7
    >>> main(n_shapes=10, min_size=4, max_size=6, seed=0)
    """
    CONFIG.update(
        dict(n_shapes=n_shapes, min_size=min_size, max_size=max_size, seed=seed)
    )

    sep = "=" * 65
    print(f"\n{sep}")
    print(
        f"  {n_shapes} polyominoes  ·  size {min_size}–{max_size}"
        f"  ·  seed={seed}  ·  MAX_BRANCHES={MAX_BRANCHES or '∞'}"
    )
    print(f"{sep}\n")

    shapes = generate_shapes(n=n_shapes, min_size=min_size, max_size=max_size, seed=seed)

    for s in shapes:
        v = get_vertices(s)
        n_p = sum(1 for x in v if x.vtype == "pointe")
        n_c = sum(1 for x in v if x.vtype == "creux")
        print(f"  {s}  |  {n_p} tip(s)  {n_c} notch(es)")
    print()

    results: dict[int, tuple] = {}
    for hi in (1, 2, 3):
        print(f"  [{_HEURISTIC_LABEL[hi]}]")
        copies = [Shape(s.id, s.cells, s.color) for s in shapes]
        best, logs, branches, trace = greedy_compact_assembly(
            copies, heuristic=hi, verbose=True
        )
        results[hi] = (best, logs, branches, trace)
        print()

    # ── Result images ─────────────────────────────────────────────────────
    print("  Generating result images …")
    visualize(shapes, results, save_path=save)

    # ── Resemblance metrics ───────────────────────────────────────────────
    base = save.replace(".png", "")

    print(f"\n  Resemblance metrics (reference: {_HEURISTIC_LABEL[3]}) …")
    metrics = compute_resemblance(results)
    for hi, m in metrics.items():
        print(
            f"    {_HEURISTIC_LABEL[hi]:<35}"
            f"  R_simple={m['simple']:.4f}"
            f"  R_weighted={m['weighted']:.4f}"
        )
    _save_resemblance(results, metrics, base + "_ressemblance.png")

    print("\n  Temporal evolution of the metrics …")
    _save_resemblance_evolution(results, base + "_ressemblance_evolution.png")

## LAUNCH

In [ ]:
if __name__ == "__main__":
    main()